# Exhibit 21 Parent–Subsidiary Pipeline: run in Google Colab

Run the cells top to bottom (**Runtime → Run all** works too).

1. Get the project (clone from GitHub, or upload the zip)
2. Install packages and run the tests
3. Build the report from the bundled Apple and Microsoft filings
4. Look at the results
5. Download more companies from SEC EDGAR: **5a** the 10 Data Scope companies, **5b** any tickers and years
6. Download the outputs

Colab runs on Google's servers with internet access, so step 5 can reach SEC EDGAR directly.

## 1. Get the project
**Option A (recommended):** run the first cell. It clones the `alina/ex21-pipeline` branch from GitHub.

**Option B:** skip the first cell, run the second one and pick `ex21_pipeline.zip` when **Choose Files** appears.

In [ ]:
import os
BRANCH = "alina/ex21-pipeline"                                     # <-- the branch with this pipeline
!rm -rf /content/KPMG-GraphRAG
!git clone -q --depth 1 -b {BRANCH} https://github.com/ZiyiClaudiaWang/KPMG-GraphRAG.git /content/KPMG-GraphRAG
os.chdir("/content/KPMG-GraphRAG/ex21_pipeline")
print("Working folder:", os.getcwd())
!ls

In [ ]:
# Option B: upload the zip instead of cloning
from google.colab import files
import zipfile, os

uploaded = files.upload()                      # choose ex21_pipeline.zip
zip_name = next(n for n in uploaded if n.endswith(".zip"))
zipfile.ZipFile(zip_name).extractall("/content")
os.chdir("/content/ex21_pipeline")
print("Working folder:", os.getcwd())
!ls

## 2. Install packages and run the format tests

In [ ]:
!pip install -q -r requirements.txt
!python tests/test_parse.py
!python tests/test_fetch.py


## 3. Build the report
Reads every Exhibit 21 in `data/raw/` and writes `output/parent_subsidiary_report.xlsx` and `output/graph_edges.csv`.

In [ ]:
!python run_pipeline.py

## 4. Look at the results
These are the same sheets as in the Excel file.

In [ ]:
import pandas as pd
report = "output/parent_subsidiary_report.xlsx"
sheets = pd.read_excel(report, sheet_name=None)
print("Sheets:", list(sheets))

print("\n=== Parent → Subsidiaries (latest year per company) ===")
display(sheets["Parent → Subsidiaries"])

In [ ]:
print("=== Year-over-year changes ===")
display(sheets["Changes"])

print("=== QA: did every file parse? ===")
display(sheets["QA"])

In [ ]:
edges = pd.read_csv("output/graph_edges.csv")
print(len(edges), "edges")
display(edges.head(10))

## 5. Download more companies from SEC EDGAR
Put **your own** name and email in `SEC_USER_AGENT` first (SEC requires a contact; it is only sent to SEC).

Already-downloaded filings are skipped, so re-running any cell here is safe. After downloading, re-run step 3 and step 4, or the cell at the end of this step.

In [ ]:
import os
os.environ["SEC_USER_AGENT"] = "Your Name your.email@school.edu"   # <-- edit this

### 5a. The 10 Data Scope companies
Fetches exactly the 10-K accessions listed in the extraction pipeline's `config/filings.yaml` on `main` (10-Q rows are skipped: only a 10-K carries Exhibit 21). Expected: one Exhibit 21 per company, 10 in all; Apple FY2025 and Microsoft FY2026 are already bundled and show as `already in manifest`.

In [ ]:
!wget -q -O /content/filings.yaml https://raw.githubusercontent.com/ZiyiClaudiaWang/KPMG-GraphRAG/main/extraction_pipeline/config/filings.yaml
!python fetch_edgar.py --from-filings /content/filings.yaml
!cat data/fetch_log.csv

### 5b. *(Optional)* Any tickers and fiscal years
- `AAPL MSFT --years 3`: the 3 most recent 10-Ks of each
- `AAPL MSFT --fy 2024 2025`: those fiscal years for every ticker
- `AAPL:2025 NKE:2024,2026`: chosen years per ticker

Fiscal year = the calendar year the 10-K's period ends (Apple's year ending 2025-09-27 is FY2025). A year with no 10-K is listed in `fetch_log.csv`.

In [ ]:
TICKERS = "NVDA:2025 JPM:2024,2025"                                 # <-- edit this
!python fetch_edgar.py {TICKERS}
!cat data/fetch_log.csv

### Rebuild and check
Every Data Scope company should appear below, and the QA sheet should list no file needing review.

In [ ]:
!python run_pipeline.py
import pandas as pd
sheets = pd.read_excel("output/parent_subsidiary_report.xlsx", sheet_name=None)
scope = ["AAPL", "CASY", "MCK", "MDT", "MSFT", "MU", "NKE", "NTAP", "PH", "TTWO"]
recs = sheets["All Records"]
counts = recs.groupby(["parent_ticker", "fiscal_year"]).size().rename("subsidiaries").reset_index()
display(counts)
print("Data Scope companies with no rows:", sorted(set(scope) - set(recs["parent_ticker"])) or "none")
display(sheets["QA"])

## 6. Download the outputs

In [ ]:
from google.colab import files
files.download("output/parent_subsidiary_report.xlsx")
files.download("output/graph_edges.csv")